### Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.

**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas11_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Feature Engineering | `StringIndexer` & `VectorAssembler` diterapkan dengan benar | 20% |
| B. Split & Pelatihan | Split proporsional benar, model berhasil dilatih | 20% |
| C. Evaluasi | RMSE & R² dihitung benar, interpretasi masuk akal | 20% |
| D. Eksperimen Perbandingan | Model kedua benar, perbandingan & kesimpulan logis | 25% |
| E. Prediksi Skenario Baru | Ketiga skenario diproses & diprediksi dengan benar | 15% |


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset CSV dengan inferSchema agar kolom numerik terbaca sesuai tipe aslinya
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)
print("Total baris dataset:", df.count())

26/11/21 19:10:06 WARN Utils: Your hostname, haris resolves to a loopback address: 127.0.1.1; using 192.168.1.57 instead (on interface wlp1s0)
26/11/21 19:10:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 19:10:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris dataset: 2500


## A. FEATURE ENGINEERING

In [4]:
# 1. StringIndexer untuk nama_menu, cabang, dan waktu_pesan
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Simpan hasil fit() agar dapat dipakai ulang untuk prediksi skenario baru (Bagian E)
model_idx_menu = indexer_menu.fit(df)
df_indexed = model_idx_menu.transform(df)

model_idx_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_idx_cabang.transform(df_indexed)

model_idx_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_idx_waktu.transform(df_indexed)

# 2. VectorAssembler untuk menggabungkan 5 fitur menjadi kolom "features"
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



## B. Train/Test Split & Pelatihan Model (bobot 20%)

In [6]:
# Pembagian data 75% training dan 25% testing (seed=101)
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

# Melatih model LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train)

print("Koefisien fitur :", model_utama.coefficients)
print("Intercept       :", model_utama.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien fitur : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept       : -66903.17766485889


## C: Evaluasi Model (Bobot 20%)

In [7]:
predictions = model_utama.transform(test)

evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
)

rmse_utama = evaluator_rmse.evaluate(predictions)
r2_utama = evaluator_r2.evaluate(predictions)

print(f"RMSE Model Utama (5 fitur) : Rp {rmse_utama:,.0f}")
print(f"R²   Model Utama (5 fitur) : {r2_utama:.4f} ({r2_utama*100:.1f}% variasi dijelaskan)")

RMSE Model Utama (5 fitur) : Rp 10,901
R²   Model Utama (5 fitur) : 0.9107 (91.1% variasi dijelaskan)


## D: Eksperimen Perbandingan Fitur (Bobot 25%)

In [8]:
# VectorAssembler kedua hanya menggunakan qty dan harga_satuan
assembler_2 = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)

df_model2 = assembler_2.transform(df)
train_2, test_2 = df_model2.randomSplit([0.75, 0.25], seed=101)

lr_2 = LinearRegression(featuresCol="features_2", labelCol="total_penjualan")
model_2 = lr_2.fit(train_2)

predictions_2 = model_2.transform(test_2)
rmse_2 = evaluator_rmse.evaluate(predictions_2)
r2_2 = evaluator_r2.evaluate(predictions_2)

print(f"RMSE Model Kedua (2 fitur) : Rp {rmse_2:,.0f}")
print(f"R²   Model Kedua (2 fitur) : {r2_2:.4f}")
print(f"Selisih R² (Utama - Kedua) : {r2_utama - r2_2:+.6f}")

RMSE Model Kedua (2 fitur) : Rp 10,896
R²   Model Kedua (2 fitur) : 0.9108
Selisih R² (Utama - Kedua) : -0.000078


## E: Prediksi Skenario Baru (Bobot 15%)

In [9]:
# Membuat 3 skenario pesanan dummy
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Cappuccino", "Yogyakarta", "Sore", 5, 25000),
    ("Es Teh", "Semarang", "Siang", 1, 15000),
]
kolom_skenario = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(data_skenario, kolom_skenario)

# Transformasi menggunakan StringIndexerModel dan VectorAssembler yang SAMA dari Bagian A
df_skenario_idx = model_idx_menu.transform(df_skenario)
df_skenario_idx = model_idx_cabang.transform(df_skenario_idx)
df_skenario_idx = model_idx_waktu.transform(df_skenario_idx)
df_skenario_final = assembler.transform(df_skenario_idx)

# Prediksi menggunakan model_utama dari Bagian B
hasil_skenario = model_utama.transform(df_skenario_final)
hasil_skenario.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

# Menutup SparkSession setelah seluruh proses selesai
spark.stop()
print("SparkSession Tugas11 ditutup.")

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |3  |20000       |59572.01525544525 |
|Cappuccino|Yogyakarta|Sore       |5  |25000       |119954.93279289985|
|Es Teh    |Semarang  |Siang      |1  |15000       |1355.5025004971103|
+----------+----------+-----------+---+------------+------------------+

SparkSession Tugas11 ditutup.
